# 18x_business_recommendation_storyline_260518

Presentation storyline and business recommendation package based on verified 17x segmentation outputs. No new modeling, SHAP recalculation, segmentation regeneration, dashboard generation, or final campaign policy decision is performed.

In [1]:
from pathlib import Path
from datetime import datetime
import hashlib, zipfile, warnings
import numpy as np
import pandas as pd
warnings.filterwarnings('ignore')

STEP = '18x_business_recommendation_storyline_260518'
START = Path.cwd().resolve()
ROOT = None
PARK = None
for cand in [START] + list(START.parents):
    if cand.name == 'park.ingyeom' and (cand / 'note.md').exists():
        PARK = cand.resolve(); ROOT = cand.parent.resolve(); break
    if (cand / 'park.ingyeom' / 'note.md').exists():
        ROOT = cand.resolve(); PARK = (cand / 'park.ingyeom').resolve(); break
assert PARK is not None and PARK.name == 'park.ingyeom', f'Could not locate park.ingyeom from {START}'

NB_PATH = PARK / 'notebook' / STEP / f'{STEP}.ipynb'
OUT = PARK / 'reports' / 'storyline' / STEP
ZIP_DIR = PARK / 'zip'
ZIP_PATH = ZIP_DIR / f'{STEP}_review_package.zip'
NOTE = PARK / 'note.md'
for p in [NB_PATH.parent, OUT, ZIP_DIR]:
    p.mkdir(parents=True, exist_ok=True)

exec_log = []
def log(msg):
    exec_log.append(f"{datetime.now().isoformat(timespec='seconds')} | {msg}")
log(f'START {STEP}')
log(f'park_root={PARK}')

def inside_park(path):
    return str(Path(path).resolve()).lower().startswith(str(PARK).lower())

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b''):
            h.update(chunk)
    return h.hexdigest()

def stat_file(path):
    p = Path(path)
    return {'sha256': sha256_file(p), 'mtime': datetime.fromtimestamp(p.stat().st_mtime).isoformat(timespec='seconds'), 'size': p.stat().st_size}

def read_csv(path):
    return pd.read_csv(path)

def write_csv(df, name):
    path = OUT / name
    df.to_csv(path, index=False, encoding='utf-8-sig')
    log(f'created {name}: rows={len(df)}, cols={len(df.columns)}')
    return path

def write_text(text, name):
    path = OUT / name
    path.write_text(text.strip() + '\n', encoding='utf-8')
    log(f'created {name}: chars={len(text)}')
    return path

def final_checks_pass(path):
    df = pd.read_csv(path)
    status_cols = [c for c in df.columns if c.lower() in {'status','result','check_status'}]
    if status_cols:
        return not df[status_cols[0]].astype(str).str.lower().str.fullmatch('fail|failed|error|critical_fail').any()
    return True

input_paths = {
    '17x_segment_summary': PARK / 'reports' / 'segments' / '17x_segmentation_design_260516' / '17x_segment_summary.csv',
    '17x_representative_segment_assignment': PARK / 'reports' / 'segments' / '17x_segmentation_design_260516' / '17x_representative_segment_assignment.csv',
    '17x_representative_segment_rules': PARK / 'reports' / 'segments' / '17x_segmentation_design_260516' / '17x_representative_segment_rules.csv',
    '17x_segment_feature_profile': PARK / 'reports' / 'segments' / '17x_segmentation_design_260516' / '17x_segment_feature_profile.csv',
    '17x_segment_SHAP_evidence_link': PARK / 'reports' / 'segments' / '17x_segmentation_design_260516' / '17x_segment_SHAP_evidence_link.csv',
    '17x_proxy_artifact_audit': PARK / 'reports' / 'segments' / '17x_segmentation_design_260516' / '17x_proxy_artifact_audit.csv',
    '17x_age40_unverified_ios_audit': PARK / 'reports' / 'segments' / '17x_segmentation_design_260516' / '17x_age40_unverified_ios_audit.csv',
    '17x_business_action_candidates': PARK / 'reports' / 'segments' / '17x_segmentation_design_260516' / '17x_business_action_candidates.csv',
    '17x_dashboard_handoff_datamart': PARK / 'reports' / 'segments' / '17x_segmentation_design_260516' / '17x_dashboard_handoff_datamart.csv',
    '17x_safe_unsafe_wording': PARK / 'reports' / 'segments' / '17x_segmentation_design_260516' / '17x_safe_unsafe_wording.csv',
    '17x_open_risks': PARK / 'reports' / 'segments' / '17x_segmentation_design_260516' / '17x_open_risks.csv',
    '17x_final_checks': PARK / 'reports' / 'segments' / '17x_segmentation_design_260516' / '17x_final_checks.csv',
}
required_cols = {
    '17x_segment_summary': ['representative_segment','row_count','row_share','repurchase_rate','mean_churn_risk'],
    '17x_representative_segment_assignment': ['row_id','USER_KEY','representative_segment','is_repurchase','repurchase_score','churn_risk','is_promotion'],
    '17x_representative_segment_rules': ['representative_segment','rule_features'],
    '17x_proxy_artifact_audit': ['representative_segment','proxy_contamination_level'],
    '17x_safe_unsafe_wording': ['wording_type','wording','reason'],
    '17x_open_risks': ['risk','handling'],
}
loaded = {}
source_before = []
pre_rows = []
for key, path in input_paths.items():
    exists = path.exists(); loaded_ok = False; row_count = None; col_count = None; cols_ok = True; status = 'PASS' if exists else 'FAIL'; detail = ''
    if exists:
        st = stat_file(path)
        source_before.append({'file_path': str(path), 'file_role': key, 'sha256_before': st['sha256'], 'mtime_before': st['mtime'], 'size_before': st['size']})
        try:
            df = pd.read_csv(path)
            loaded[key] = df
            loaded_ok = True; row_count = len(df); col_count = len(df.columns)
            cols_ok = all(c in df.columns for c in required_cols.get(key, []))
            if not cols_ok:
                status = 'FAIL'; detail = 'required columns missing'
            if key == '17x_final_checks' and not final_checks_pass(path):
                status = 'FAIL'; detail = '17x final_checks contains FAIL'
        except Exception as e:
            status = 'FAIL'; detail = repr(e)
    pre_rows.append({'input_file': path.name, 'path': str(path), 'exists': exists, 'loaded': loaded_ok, 'row_count': row_count, 'column_count': col_count, 'required_columns_present': cols_ok, 'status': status, 'detail': detail})
preflight = pd.DataFrame(pre_rows)
write_csv(preflight, '18x_preflight_input_validation.csv')

summary = loaded.get('17x_segment_summary', pd.DataFrame()).copy()
assignment = loaded.get('17x_representative_segment_assignment', pd.DataFrame()).copy()
rules = loaded.get('17x_representative_segment_rules', pd.DataFrame()).copy()
profile = loaded.get('17x_segment_feature_profile', pd.DataFrame()).copy()
shap_link = loaded.get('17x_segment_SHAP_evidence_link', pd.DataFrame()).copy()
proxy = loaded.get('17x_proxy_artifact_audit', pd.DataFrame()).copy()
age_audit = loaded.get('17x_age40_unverified_ios_audit', pd.DataFrame()).copy()
actions17 = loaded.get('17x_business_action_candidates', pd.DataFrame()).copy()
dashboard = loaded.get('17x_dashboard_handoff_datamart', pd.DataFrame()).copy()
safe17 = loaded.get('17x_safe_unsafe_wording', pd.DataFrame()).copy()
risks17 = loaded.get('17x_open_risks', pd.DataFrame()).copy()
seg_count_ok = len(summary['representative_segment'].dropna().unique()) == 7 if len(summary) else False
assign_count_ok = len(assignment) == 23079
one_seg_ok = assign_count_ok and assignment['representative_segment'].notna().all() and len(assignment) == assignment['row_id'].nunique()
log(f'preflight 17x_pass={final_checks_pass(input_paths["17x_final_checks"])} seg_count_ok={seg_count_ok} assignment_rows={len(assignment)} one_seg_ok={one_seg_ok}')

summary['row_count'] = pd.to_numeric(summary['row_count'])
summary['row_share'] = pd.to_numeric(summary['row_share'])
summary['repurchase_rate'] = pd.to_numeric(summary['repurchase_rate'])
summary['mean_churn_risk'] = pd.to_numeric(summary['mean_churn_risk'])
seg_stats = {r['representative_segment']: r for _, r in summary.iterrows()}

signal_map = {
    'high_risk_week3_inactive_or_drop': 'week3 inactive/drop and retention decay signal in day0-20 window',
    'high_risk_only_w1_or_cold_start_weak': 'weak early activation or week1-only pattern',
    'high_risk_low_activity': 'low total activity with high churn_risk score',
    'medium_risk_retention_decay': 'not top20 high risk, but week-to-week retention decay is observed',
    'content_preference_target_candidate': 'content or genre proxy is comparatively clearer while low-activity is not dominant',
    'stable_retained_user': 'low risk and stable retained behavior signal',
    'general_observation': 'no priority behavior rule strongly matched'
}
problem_map = {
    'high_risk_week3_inactive_or_drop': 'late observation-window cooling signal may need quick re-entry support',
    'high_risk_only_w1_or_cold_start_weak': 'initial viewing habit may not have formed',
    'high_risk_low_activity': 'low engagement makes high-intensity personalization risky',
    'medium_risk_retention_decay': 'risk is moderate, but decay can be detected before it becomes severe',
    'content_preference_target_candidate': 'recommendation can use content proxy, with mapping caveat',
    'stable_retained_user': 'defensive discount may be inefficient for already stable rows',
    'general_observation': 'needs monitoring or additional information rather than a sharp claim'
}
action_map = {
    'high_risk_week3_inactive_or_drop': 'day21 직후 재진입 알림, 최근 관측창 내 선호 기반 짧은 추천, 시청 중단 장르의 유사 콘텐츠 리마인드 후보',
    'high_risk_only_w1_or_cold_start_weak': '온보딩 보강, 첫 시청 또는 두 번째 시청 유도, 짧은 길이와 낮은 진입장벽 콘텐츠 추천 후보',
    'high_risk_low_activity': '과도한 할인보다 낮은 마찰의 재방문 유도와 broad recommendation 후보',
    'medium_risk_retention_decay': 'week2-week3 감소 감지 후 day21 이전 또는 직후 유지 메시지 후보',
    'content_preference_target_candidate': 'Movie_Master category mapping 기준 장르/콘텐츠 기반 추천 후보',
    'stable_retained_user': '방어성 할인보다 만족도 유지, 업셀, 리뷰/추천 유도 후보',
    'general_observation': '일반 모니터링 또는 추가 데이터 확보 후보'
}
timing_map = {
    'high_risk_week3_inactive_or_drop': 'day21 직후', 'high_risk_only_w1_or_cold_start_weak': '초기 이용 후 day7~day21 사이 또는 day21 직후', 'high_risk_low_activity': 'day21 직후 낮은 빈도 리마인드', 'medium_risk_retention_decay': 'week2~week3 감소 감지 후 day21 전후', 'content_preference_target_candidate': 'day21 이후 추천 슬롯', 'stable_retained_user': '정기 만족도/업셀 접점', 'general_observation': '정기 모니터링'
}

storyline = f'''
# 18x Business Recommendation Storyline

## 1. 문제 제기
100원딜 프로모션 유입 row는 단순히 재구매율만 비교해서 판단하기보다, day0~20 관측창 안에서 어떤 행동 신호를 보였는지를 기준으로 이탈 방어 전략을 설계해야 한다. 이 단계의 목적은 고객 개인을 단정하는 것이 아니라 subscription-event rows에서 관측된 행동 신호를 바탕으로 대응 후보를 정리하는 것이다.

## 2. 분석 설계
분석 시간축은 day0~20 관측, day21 scoring point, 이후 대응기간으로 둔다. target은 다음 달 재구매 여부인 `is_repurchase`이며, `repurchase_score = P(is_repurchase=1)`, `churn_risk = 1 - repurchase_score`로 해석한다. 분석 단위는 고객 수가 아니라 row-level subscription-event rows다.

## 3. 모델 결과의 역할
모델은 고객을 단정하는 도구가 아니라 이탈 위험과 행동 패턴을 묶어 대응 우선순위를 정하는 보조 도구다. SHAP은 원인이 아니라 fitted model이 어떤 변수를 중요하게 사용했는지에 대한 설명이다.

## 4. Segment 전환
17x에서는 high risk, retention decay, low activity, content preference, stable retained 등 행동 기반 provisional representative segment를 만들었다. 총 7개 segment이며, representative assignment는 23,079 subscription-event rows에 대해 row당 하나씩 부여되었다.

## 5. 비즈니스 제언
segment별로 다른 대응이 필요하다. 모든 row에 같은 할인 또는 알림을 보내기보다, week3 inactive/drop, 초기 activation 약화, 저활동, retention decay, content preference proxy, stable retained 상태에 따라 메시지와 타이밍을 다르게 가져간다.

## 6. 주의점
payment/auth/demographic proxy를 제언 근거로 쓰지 않는다. SHAP은 인과가 아니다. 100원딜 효과도 인과가 아니다. 이 결과는 row-level 분석이며 unique customer 수로 표현하지 않는다.

## 7. 결론
100원딜 이탈 방어의 핵심은 '싸게 들어온 고객을 붙잡자'가 아니라, day0~20 안에서 식어가는 신호를 조기에 발견하고 day21 이후 대응기간에 맞춤 개입하자는 것이다.
'''
write_text(storyline, '18x_storyline_master.md')

slide_rows = [
    (1,'프로젝트 질문','100원딜 유입 이후 재구매를 어떻게 방어할 것인가','target=is_repurchase, score=repurchase_score/churn_risk','time axis diagram','고객 단정이 아니라 subscription-event row의 행동 신호를 본다','subscription-event rows 기준 분석','고객 수라고 말하지 않는다'),
    (2,'왜 단순 재구매율 비교만으로 부족한가','같은 재구매율 뒤에도 행동 패턴은 다르다','17x segment별 row_count와 repurchase_rate','segment summary table','전체 평균보다 segment별 대응 가능성을 봐야 한다','행동 신호별 대응 후보','100원딜 효과를 인과로 말하지 않는다'),
    (3,'분석 시간축','day0~20 관측, day21 이후 대응 후보','17x open risks and safe wording','timeline','day21 이후 행동은 feature로 쓰지 않았다','day0~20 관측창 내 행동 신호','4주차까지 보고 판단했다고 말하지 않는다'),
    (4,'모델과 segment의 역할','모델은 대응 우선순위 보조 도구다','17x score source and SHAP evidence link','model role diagram','SHAP은 모델 설명이고 segment는 provisional이다','model explanation','SHAP causal claim 금지'),
    (5,'17x 대표 segment 분포','7개 provisional segment로 row당 하나 배정했다','17x_segment_summary.csv','bar chart','row count 기준 분포를 먼저 보여준다','row-level representative segment','final 고객 유형이라고 말하지 않는다'),
    (6,'high_risk_week3_inactive_or_drop','week3 inactive/drop 고위험 row는 day21 직후 재진입 후보','row_count and mean_churn_risk','segment card','이탈 확정이 아니라 위험 후보다','이탈 위험 후보','이 고객은 이탈한다 금지'),
    (7,'high_risk_only_w1_or_cold_start_weak','초기 activation 약화 row는 온보딩 보강 후보','row_count and behavior rule','segment card','첫 시청/두 번째 시청 유도 후보로 말한다','초기 activation 약화','고객 성향 단정 금지'),
    (8,'high_risk_low_activity','저활동 고위험 row는 낮은 마찰의 재방문 유도 후보','row_count and mean_churn_risk','segment card','과도한 개인화보다 broad recommendation이 안전하다','낮은 활동량 신호','할인 확정 금지'),
    (9,'medium_risk_retention_decay','중위험 retention decay row는 조기 유지 메시지 후보','row_count and retention decay rule','segment card','고위험 전 단계의 감소 신호로 제시한다','주차별 사용 감소','인과 표현 금지'),
    (10,'content_preference_target_candidate','콘텐츠 proxy가 뚜렷한 row는 유사 장르 추천 후보','row_count and content proxy caveat','segment card','Movie_Master category mapping proxy라고 caveat를 붙인다','콘텐츠 취향 proxy','정확한 취향이라고 단정 금지'),
    (11,'stable_retained_user','안정 row는 방어성 할인보다 유지/업셀 후보','repurchase_rate and mean_churn_risk','segment card','할인을 줄인다는 결론이 아니라 후보로 말한다','안정적 재구매 가능성 후보','최종 정책 확정 금지'),
    (12,'proxy/artifact 주의점','payment/auth/demographic proxy는 근거가 아니라 audit이다','17x_proxy_artifact_audit.csv','caution table','proxy를 segment 이름에 쓰지 않는다','proxy audit','iOS/40대/미인증 세그먼트 금지'),
    (13,'최종 비즈니스 운영안','행동 신호별 메시지와 타이밍을 다르게 설계한다','18x recommendation matrix','matrix','제언은 campaign candidate다','제언 후보','A/B test 전 정책 확정 금지'),
    (14,'한계와 다음 단계','검증과 실험 설계가 다음 단계다','18x_open_risks.csv','risk list','A/B test와 dashboard handoff로 이어진다','남은 리스크 명시','확정 표현 금지'),
]
slide_outline = pd.DataFrame(slide_rows, columns=['slide_no','slide_title','main_message','supporting_evidence','recommended_visual','speaker_note','safe_wording','risk_or_caution'])
write_csv(slide_outline, '18x_slide_outline.csv')

matrix_rows = []
for _, r in summary.sort_values('segment_priority').iterrows():
    seg = r['representative_segment']
    matrix_rows.append({
        'representative_segment': seg, 'row_count': int(r['row_count']), 'row_share': r['row_share'], 'repurchase_rate': r['repurchase_rate'], 'mean_churn_risk': r['mean_churn_risk'],
        'observed_behavior_signal': signal_map.get(seg,''), 'business_problem': problem_map.get(seg,''), 'recommended_action_candidate': action_map.get(seg,''), 'message_timing': timing_map.get(seg,''),
        'message_type': 'behavior-based candidate message', 'success_metric_candidate': 'next-cycle repurchase, day21+ re-entry, watch activity recovery, message engagement',
        'do_not_say': 'Do not say this customer will churn, SHAP is causal, 100-won-deal caused churn, or proxy variables define the segment.',
        'caution': 'Campaign candidate only; A/B test required; row-level subscription-event analysis.'
    })
recommendation_matrix = pd.DataFrame(matrix_rows)
write_csv(recommendation_matrix, '18x_business_recommendation_matrix.csv')

message_rows = []
for _, r in recommendation_matrix.iterrows():
    seg = r['representative_segment']
    message_rows.append({
        'representative_segment': seg,
        'primary_message_goal': problem_map.get(seg,''),
        'message_example_safe': '최근 관측창 내 행동 흐름이 줄어든 subscription-event row에 대해, 이전 관측창 내 선호 콘텐츠 proxy와 유사한 콘텐츠를 추천한다.' if 'content' in seg or 'inactive' in seg or 'decay' in seg else '관측창 내 행동 신호를 기준으로 낮은 마찰의 재방문 후보 메시지를 제안한다.',
        'message_example_unsafe': 'iOS 결제 고객은 충성도가 높으므로 업셀한다. 40대 미인증 고객은 특정 세그먼트다.',
        'timing_candidate': r['message_timing'],
        'channel_candidate': 'app push / in-app message / email candidate, final channel not fixed',
        'personalization_basis': 'day0~20 observed behavior and content proxy only, not payment/auth/demographic proxy',
        'why_this_is_safe': 'It uses observed behavior signals and keeps the recommendation as a candidate, not a causal or final policy claim.'
    })
write_csv(pd.DataFrame(message_rows), '18x_segment_to_message_strategy.csv')

script = '''# 18x Presentation Narrative Script

본 분석은 고객 개인을 단정하는 것이 아니라, subscription-event row에서 관측된 행동 신호를 바탕으로 대응 후보를 설계하는 것입니다.

먼저 분석 시간축은 day0~20 관측창, day21 scoring point, 이후 대응기간으로 나누었습니다. target은 다음 달 재구매 여부이며, 모델의 repurchase_score는 재구매 가능성 점수, churn_risk는 1에서 repurchase_score를 뺀 값으로 사용했습니다.

모델은 고객을 확정적으로 분류하기 위한 도구가 아니라, 어떤 row에서 이탈 위험이 높게 나타나고 어떤 행동 신호가 함께 보이는지 정리하기 위한 보조 도구입니다. SHAP은 원인이 아니라 모델이 어떤 변수를 중요하게 사용했는지에 대한 설명입니다.

17x segmentation에서는 payment/auth/demographic 변수는 대표 세그먼트 기준으로 쓰지 않았습니다. payment/auth/demographic 변수는 해석 리스크가 있어 대표 세그먼트 기준으로 쓰지 않았습니다.

100원딜 여부는 집단 차이로 해석하며, 인과효과로 단정하지 않습니다. 따라서 발표에서는 100원딜이 이탈을 유발했다고 말하지 않고, 관측된 집단과 행동 신호의 차이를 바탕으로 대응 후보를 제안합니다.

비즈니스 제언은 모든 row에 동일한 할인이나 알림을 보내는 것이 아니라, week3 inactive/drop, 초기 activation 약화, low activity, retention decay, content preference proxy, stable retained 상태에 따라 메시지와 타이밍을 다르게 설계하는 방향입니다.

결론적으로 100원딜 이탈 방어의 핵심은 싸게 들어온 고객을 붙잡자는 단순 메시지가 아니라, day0~20 안에서 식어가는 신호를 조기에 발견하고 day21 이후 대응기간에 맞춤 개입하자는 것입니다.
'''
write_text(script, '18x_presentation_narrative_script.md')

qa_rows = [
    ('왜 고객 수가 아니라 row 수라고 하나요?','unit risk','이 데이터는 subscription-event row 단위로 구성되어 있고 USER_KEY 중복이 있을 수 있으므로 unique customer 수로 표현하지 않습니다.','17x_representative_segment_assignment.csv','고객 23,079명입니다.'),
    ('100원딜이 이탈을 유발했다고 볼 수 있나요?','causal risk','아니요. 100원딜 여부는 관측된 집단 차이로만 다루며 인과효과로 단정하지 않습니다.','17x_safe_unsafe_wording.csv','100원딜 때문에 이탈했습니다.'),
    ('SHAP이 높으면 원인 아닌가요?','SHAP causal risk','SHAP은 fitted model explanation입니다. 모델이 어떤 변수를 중요하게 사용했는지를 설명할 뿐 원인을 증명하지 않습니다.','17x_segment_SHAP_evidence_link.csv','SHAP이 원인을 밝혔습니다.'),
    ('payment_is_ios를 왜 제거했나요?','proxy risk','payment_device는 시청기기가 아니라 결제기기/결제환경 proxy이므로 해석 리스크가 큽니다. 15x sensitivity와 사용자 승인에 따라 SHAP input에서 제거되었습니다.','16x_payment_removed_input_gate.csv','iOS 고객 특성을 제거했습니다.'),
    ('40대 미인증 iOS 조합은 고객 특성 아닌가요?','demographic proxy risk','이 조합은 audit flag로만 관리하며 대표 세그먼트 기준이나 제언 근거로 쓰지 않습니다.','17x_age40_unverified_ios_audit.csv','40대 미인증 iOS 고객군입니다.'),
    ('segment 이름이 너무 임의적인 것 아닌가요?','label risk','맞습니다. 17x segment 이름은 provisional label이며 final customer type이 아닙니다. priority rule과 행동 조건을 함께 제시해야 합니다.','17x_representative_segment_rules.csv','최종 고객 유형입니다.'),
    ('이 결과로 바로 캠페인을 실행해도 되나요?','policy risk','아니요. 이번 결과는 campaign candidate이며 A/B test와 운영 검증이 필요합니다.','18x_open_risks.csv','바로 집행하면 됩니다.'),
    ('day21 이후 행동은 왜 feature로 쓰지 않았나요?','leakage risk','day21은 scoring point이므로 이후 행동을 feature로 쓰면 대응 전 예측이 아니라 사후 정보를 보는 문제가 생깁니다.','17x_open_risks.csv','4주차까지 보고 판단했습니다.'),
    ('genre 추천은 정확한 취향이라고 봐도 되나요?','content proxy risk','장르 변수는 Movie_Master category mapping 기준 proxy입니다. 정확한 취향 확정이 아니라 추천 후보 근거로만 사용합니다.','17x_segment_feature_profile.csv','정확한 취향입니다.'),
    ('stable_retained_user에게 할인해야 하나요?','action risk','방어성 할인보다 만족도 유지, 업셀, 리뷰/추천 유도 후보가 더 안전합니다. 단 최종 정책은 실험으로 검증해야 합니다.','18x_business_recommendation_matrix.csv','할인을 주면 됩니다.'),
]
write_csv(pd.DataFrame(qa_rows, columns=['question','risk_type','safe_answer','evidence_file','do_not_answer_like_this']), '18x_mentor_QA_defense.csv')

safe_unsafe_rows = [
    ('unsafe','고객 수','unique customer count not verified'),('unsafe','iOS 고객은 충성도가 높다','payment proxy misuse'),('unsafe','40대 미인증 iOS 고객군','proxy segment naming forbidden'),('unsafe','SHAP이 원인이다','SHAP is model explanation only'),('unsafe','100원딜이 이탈을 유발했다','causal claim forbidden'),('unsafe','payment_device는 시청기기다','field meaning is payment environment proxy'),('unsafe','4주차까지 보고 판단했다','day0~20 observation window violation'),('unsafe','이 고객은 이탈한다','deterministic individual claim forbidden'),('unsafe','이 segment가 최종 고객 유형이다','segment is provisional'),
    ('safe','subscription-event rows','row-level wording'),('safe','day0~20 관측창 내 행동 신호','observation-window wording'),('safe','재구매 가능성 점수','repurchase_score safe wording'),('safe','이탈 위험 후보','non-deterministic risk wording'),('safe','모델 설명','SHAP safe wording'),('safe','제언 후보','not final policy'),('safe','provisional representative segment','not final customer type'),('safe','Movie_Master category mapping 기준 콘텐츠 취향 proxy','content caveat')
]
write_csv(pd.DataFrame(safe_unsafe_rows, columns=['wording_type','wording','reason']), '18x_safe_unsafe_wording.csv')

onepager = f'''# 18x Onepager

## 문제
100원딜 프로모션 유입 이후 재구매 방어는 단순 재구매율 비교가 아니라 day0~20 관측창 내 행동 신호 기반 대응 설계가 필요하다.

## 분석 설계
day0~20 관측, day21 scoring point, 이후 대응기간으로 나누고, target은 다음 달 재구매 여부로 둔다. 분석 단위는 subscription-event rows다.

## 핵심 발견
17x에서 7개 provisional representative segment가 생성되었고, 고위험 inactive/drop, 초기 activation 약화, 저활동, retention decay, content proxy, stable retained 등 행동 기반 대응 후보가 분리되었다.

## 대표 Segment
가장 큰 segment는 general_observation과 content_preference_target_candidate이며, 고위험 대응 후보로는 high_risk_week3_inactive_or_drop, high_risk_only_w1_or_cold_start_weak, high_risk_low_activity가 있다.

## 제언
day21 이후 동일한 메시지를 보내기보다 행동 신호별로 재진입, 온보딩, broad recommendation, retention decay 대응, content proxy 추천, 안정 row 유지/업셀 후보를 분리한다.

## 주의점
SHAP은 인과가 아니며, 100원딜 여부도 인과로 해석하지 않는다. payment/auth/demographic proxy는 제언 근거로 직접 쓰지 않는다.

## 한 줄 결론
100원딜 이탈 방어의 핵심은 day0~20 안에서 식어가는 신호를 조기에 발견하고 day21 이후 대응기간에 맞춤 개입하는 것이다.
'''
write_text(onepager, '18x_storyline_onepager.md')

priority_order = ['high_risk_week3_inactive_or_drop','medium_risk_retention_decay','high_risk_only_w1_or_cold_start_weak','high_risk_low_activity','content_preference_target_candidate','stable_retained_user','general_observation']
priority_rows = []
for i, seg in enumerate(priority_order, 1):
    r = seg_stats[seg]
    priority_rows.append({'presentation_priority': i, 'representative_segment': seg, 'why_it_matters': signal_map[seg], 'business_actionability': action_map[seg], 'risk_level': 'high' if str(seg).startswith('high_risk') else ('medium' if 'medium' in seg else 'contextual'), 'recommended_slide_position': 5 + i, 'row_count': int(r['row_count']), 'mean_churn_risk': r['mean_churn_risk'], 'caution': 'Priority is based on actionability, not row_count alone; segment label is provisional.'})
write_csv(pd.DataFrame(priority_rows), '18x_segment_priority_for_presentation.csv')

open_risks = pd.DataFrame([
    ('segment 이름은 provisional','사용자 승인 전 final customer type으로 부르지 않는다.'),('제언은 campaign candidate','A/B test 전 최종 정책으로 쓰지 않는다.'),('A/B test 필요','운영 효과는 실험으로 검증해야 한다.'),('SHAP은 인과 아님','model explanation으로만 사용한다.'),('100원딜 효과는 인과 아님','관측된 집단 차이로만 말한다.'),('row-level 분석','고객 수 또는 unique customer 수로 말하지 않는다.'),('payment/auth/demographic proxy caution','제언 근거로 직접 사용하지 않는다.'),('genre/content mapping caveat','Movie_Master category mapping 기준 proxy다.'),('day21 이후 행동 미사용','leakage 방지를 위한 설계이며 이후 행동으로 사후 판단하지 않는다.')
], columns=['risk','handling'])
write_csv(open_risks, '18x_open_risks.csv')

source_after = []
for row in source_before:
    p = Path(row['file_path']); st = stat_file(p)
    source_after.append({**row, 'sha256_after': st['sha256'], 'mtime_after': st['mtime'], 'size_after': st['size'], 'status': 'PASS' if row['sha256_before'] == st['sha256'] and row['size_before'] == st['size'] else 'FAIL'})
fingerprint = pd.DataFrame(source_after)
write_csv(fingerprint, '18x_source_fingerprint_before_after.csv')

readme = f'''# {STEP}

## Purpose
18x converts verified 17x segmentation outputs into presentation-ready business recommendation storyline artifacts. It does not perform new modeling, SHAP recalculation, segmentation regeneration, dashboard generation, or final campaign policy selection.

## Inputs
The package reads 17x segment summary, assignment, rules, feature profile, SHAP evidence link, proxy audit, age40-unverified-iOS audit, action candidates, dashboard handoff, safe/unsafe wording, open risks, and final checks. `17x_final_checks.csv` is verified as PASS.

## How to Use
Use `18x_storyline_master.md` and `18x_presentation_narrative_script.md` for the main presentation flow, `18x_slide_outline.csv` for slide planning, and `18x_mentor_QA_defense.csv` for backup defense logic.

## Interpretation Limits
Use subscription-event rows, not customer count. Segment labels are provisional representative segments. Recommendations are campaign candidates and require A/B testing. SHAP is model explanation, not causal evidence. 100-won-deal differences are observed group differences, not causal effects. Payment/auth/demographic proxies are not recommendation bases.

## Remaining Risks
See `18x_open_risks.csv`. Main risks are provisional segment names, need for A/B test, row-level interpretation, content mapping proxy caveat, and exclusion of day21+ behavior from features.
'''
write_text(readme, 'README.md')

marker = '## 2026-05-18 | 18x_business_recommendation_storyline_260518 completion'
note_text = NOTE.read_text(encoding='utf-8')
if marker not in note_text:
    append = f'''

{marker}

18x_business_recommendation_storyline_260518을 수행했다. 이번 18x는 발표용 비즈니스 제언 스토리라인 정리 단계이며, 새 모델링, SHAP 재계산, segmentation 재생성, dashboard 제작, 캠페인 정책 최종 확정 단계가 아니다.

17x representative segment 결과를 발표 흐름으로 변환했다. segment 이름은 provisional이며 사용자 승인 전 final customer type으로 부르지 않는다. 제언은 campaign candidate이며 A/B test 전 최종 정책이 아니다.

payment/auth/demographic proxy는 제언 근거로 직접 사용하지 않았다. SHAP은 인과가 아니라 model explanation으로만 사용했고, 100원딜 여부는 인과가 아니라 관측된 집단 차이로만 해석하도록 안전 문구를 정리했다. 분석 단위는 row-level/subscription-event-level이며 고객 수 또는 unique customer 수로 말하면 안 된다.

생성 산출물: 18x_preflight_input_validation.csv, 18x_storyline_master.md, 18x_slide_outline.csv, 18x_business_recommendation_matrix.csv, 18x_segment_to_message_strategy.csv, 18x_presentation_narrative_script.md, 18x_mentor_QA_defense.csv, 18x_safe_unsafe_wording.csv, 18x_storyline_onepager.md, 18x_segment_priority_for_presentation.csv, 18x_open_risks.csv, 18x_source_fingerprint_before_after.csv, 18x_final_checks.csv, README.md, 18x_execution_log.txt, note_tail_copy.md, 18x_review_zip_inventory.csv, review zip.

다음 단계 인수인계: 발표 슬라이드 제작 시 `18x_slide_outline.csv`를 기본 목차로 쓰고, 본문 문장은 `18x_presentation_narrative_script.md`와 `18x_safe_unsafe_wording.csv`의 안전 표현만 사용한다. 멘토 질문 방어는 `18x_mentor_QA_defense.csv`를 기준으로 한다.
'''
    NOTE.write_text(note_text.rstrip() + append + '\n', encoding='utf-8')
log('updated note.md')
(OUT / 'note_tail_copy.md').write_text('\n'.join(NOTE.read_text(encoding='utf-8').splitlines()[-220:]) + '\n', encoding='utf-8')

checks = []
def add_check(name, cond, detail=''):
    checks.append({'check_name': name, 'status': 'PASS' if bool(cond) else 'FAIL', 'detail': detail})
outputs_now = list(OUT.glob('*')) + [NB_PATH, ZIP_PATH]
matrix_text = ' '.join(recommendation_matrix.astype(str).values.ravel()).lower()
safe_text = ' '.join(pd.read_csv(OUT / '18x_safe_unsafe_wording.csv').astype(str).values.ravel()).lower()
add_check('all_outputs_inside_park_ingyeom', all(inside_park(p) for p in outputs_now), '')
add_check('notebook_exists', NB_PATH.exists(), str(NB_PATH))
add_check('notebook_executed', True, 'execution reached final checks cell')
add_check('17x_inputs_loaded', preflight['status'].eq('PASS').all(), '')
add_check('17x_final_checks_pass', final_checks_pass(input_paths['17x_final_checks']), '')
add_check('segment_summary_loaded', len(summary) == 7 and seg_count_ok, '')
add_check('representative_assignment_row_count_23079', assign_count_ok and one_seg_ok, '')
add_check('storyline_master_created', (OUT / '18x_storyline_master.md').exists(), '')
add_check('slide_outline_created', (OUT / '18x_slide_outline.csv').exists(), '')
add_check('business_recommendation_matrix_created', (OUT / '18x_business_recommendation_matrix.csv').exists(), '')
add_check('message_strategy_created', (OUT / '18x_segment_to_message_strategy.csv').exists(), '')
add_check('presentation_script_created', (OUT / '18x_presentation_narrative_script.md').exists(), '')
add_check('mentor_QA_created', (OUT / '18x_mentor_QA_defense.csv').exists(), '')
add_check('safe_unsafe_wording_created', (OUT / '18x_safe_unsafe_wording.csv').exists(), '')
add_check('onepager_created', (OUT / '18x_storyline_onepager.md').exists(), '')
add_check('open_risks_created', (OUT / '18x_open_risks.csv').exists(), '')
add_check('source_fingerprint_created', (OUT / '18x_source_fingerprint_before_after.csv').exists(), '')
add_check('source_fingerprint_unchanged', fingerprint['status'].eq('PASS').all(), '')
add_check('no_model_training_performed', True, '18x reads 17x outputs only')
add_check('no_SHAP_recalculation_performed', True, '18x reads 17x SHAP evidence link only')
add_check('no_segmentation_regeneration_performed', True, '18x uses 17x representative assignment')
add_check('no_dashboard_generation_performed', True, '18x creates storyline handoff documents only')
add_check('no_payment_proxy_used_as_recommendation_basis', 'payment_is_' not in matrix_text and 'payment_device' not in matrix_text, '')
add_check('no_auth_proxy_used_as_recommendation_basis', 'is_user_verified' not in matrix_text and '미인증' not in matrix_text, '')
add_check('no_demographic_proxy_used_as_recommendation_basis', 'age_group' not in matrix_text and '40대' not in matrix_text, '')
add_check('SHAP_wording_marked_as_model_explanation', 'shap은 원인이 아니라' in (OUT / '18x_storyline_master.md').read_text(encoding='utf-8').lower() or 'shap is model explanation' in (OUT / 'README.md').read_text(encoding='utf-8').lower(), '')
add_check('causal_claims_blocked', '100원딜이 이탈을 유발했다' in safe_text and 'unsafe' in safe_text, '')
add_check('row_level_wording_used', 'subscription-event rows' in (OUT / '18x_storyline_master.md').read_text(encoding='utf-8'), '')
add_check('README_created', (OUT / 'README.md').exists(), '')
add_check('note_md_updated', marker in NOTE.read_text(encoding='utf-8'), '')
add_check('review_zip_created', False, 'created after final_checks then refreshed')
add_check('review_zip_inventory_created', False, 'created after final_checks then refreshed')
final_checks = pd.DataFrame(checks)
write_csv(final_checks, '18x_final_checks.csv')
log(f'initial final_checks fail_count={int(final_checks.status.eq("FAIL").sum())}')

log('END notebook execution before packaging')
(OUT / '18x_execution_log.txt').write_text('\n'.join(exec_log) + '\n', encoding='utf-8')
review_items = []
for p in [NB_PATH] + sorted(OUT.glob('*')):
    if p.exists() and p.is_file():
        review_items.append({'path': str(p), 'arcname': str(p.relative_to(PARK)), 'size_bytes': p.stat().st_size})
review_inv = pd.DataFrame(review_items)
review_inv.to_csv(OUT / '18x_review_zip_inventory.csv', index=False, encoding='utf-8-sig')
with zipfile.ZipFile(ZIP_PATH, 'w', zipfile.ZIP_DEFLATED) as z:
    for item in review_items:
        z.write(item['path'], item['arcname'])
    z.write(OUT / '18x_review_zip_inventory.csv', str((OUT / '18x_review_zip_inventory.csv').relative_to(PARK)))
log(f'created review zip {ZIP_PATH} size={ZIP_PATH.stat().st_size}')

final_checks.loc[final_checks['check_name'].eq('review_zip_created'), ['status','detail']] = ['PASS', str(ZIP_PATH)]
final_checks.loc[final_checks['check_name'].eq('review_zip_inventory_created'), ['status','detail']] = ['PASS', str(OUT / '18x_review_zip_inventory.csv')]
write_csv(final_checks, '18x_final_checks.csv')
note_text = NOTE.read_text(encoding='utf-8')
if '18x final_checks 결과' not in note_text:
    NOTE.write_text(note_text.rstrip() + f"\n\n18x final_checks 결과는 `{int(final_checks.status.eq('PASS').sum())} PASS / {int(final_checks.status.eq('FAIL').sum())} FAIL`이다. review zip은 `zip/18x_business_recommendation_storyline_260518_review_package.zip`에 생성했다.\n", encoding='utf-8')
(OUT / 'note_tail_copy.md').write_text('\n'.join(NOTE.read_text(encoding='utf-8').splitlines()[-220:]) + '\n', encoding='utf-8')
(OUT / '18x_execution_log.txt').write_text('\n'.join(exec_log) + '\n', encoding='utf-8')
print('18x complete')
print(final_checks['status'].value_counts().to_dict())
print('zip', ZIP_PATH)


18x complete
{'PASS': 32}
zip C:\Code\ott-churn-prediction\park.ingyeom\zip\18x_business_recommendation_storyline_260518_review_package.zip
